<a href="https://colab.research.google.com/github/enm0910/ST554/blob/main/EMartinez_Project3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Author: Emma Martinez

Course: ST 554 Project 3

Purpose: Project 3- PySpark ML + Streaming Project

# Project #3

This notebook has two parts:

**Part 1** – Fits an elastic net regression model to predict Power_Zone_3 (power consumption in Tetouan City) using PySpark's MLlib. We load the data, build a transformation pipeline (binarization, one-hot encoding, PCA), and tune the model via 5-fold cross-validation across a grid of regularization parameters.

**Part 2** – Reads a live stream of incoming CSV files using PySpark Structured Streaming. We apply the fitted model to each batch of arriving data to generate real-time predictions, compute residuals, and write the results to the console.

We start every section by installing PySpark and creating (or reusing) a SparkSession.

### Setup

In [15]:
# Install PySpark for every session
!pip install pyspark

In [25]:
# Mount Google Drive so notebooks can share stream_input/folder
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/MyDrive/ST554Project3"
os.makedirs(f"{BASE}/stream_input", exist_ok=True)

Mounted at /content/drive


In [16]:
# Imports
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import (StructType, StructField,
                                DoubleType, IntegerType, StringType)
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    Binarizer, OneHotEncoder, StringIndexer,
    VectorAssembler, PCA, SQLTransformer
)
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

In [17]:
# Start a local Spark session
spark = SparkSession.builder \
    .appName("PowerZone3Prediction") \
    .getOrCreate()

# Surpress normal Spark logs
spark.sparkContext.setLogLevel("ERROR")
print("Spark session started")

Spark session started


### Part 1: Model Fitting

#### 1.1: Load data

In [18]:
# Read csv into pandas and convert to Spark
pandas_df = pd.read_csv("/content/power_ml_data.csv")

# Cast Month to string so StringIndexer works right
pandas_df["Month"] = pandas_df["Month"].astype(str)

spark_df = spark.createDataFrame(pandas_df)

spark_df.printSchema()
spark_df.show(5)

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: string (nullable = true)
 |-- Hour: long (nullable = true)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375

####1.2: Define pipline stages

In [19]:
# Stage 1: Cast Hour to doubletype (for binarizer)
sql_transform = SQLTransformer(
    statement="SELECT *, CAST(Hour AS DOUBLE) AS Hour_dbl FROM __THIS__"
)

# Stage 2: Binarize Hour- 0 if night (< 6.5), 1 if day (>= 6.5)
binarizer = Binarizer(
    threshold=6.5,
    inputCol="Hour_dbl",
    outputCol="Hour_binary"
)

# Stage 3 & 4: Index then one-hot encode Month
month_indexer = StringIndexer(inputCol="Month", outputCol="Month_index")
month_encoder = OneHotEncoder(inputCols=["Month_index"], outputCols=["Month_ohe"])

# Stage 5: Assemble weather columns into a vector for PCA
weather_cols = ["Temperature", "Humidity", "Wind_Speed",
                "General_Diffuse_Flows", "Diffuse_Flows"]

weather_assembler = VectorAssembler(
    inputCols=weather_cols,
    outputCol="weather_features"
)

# Stage 6: PCA— reduce weather columns to 2 principal components
pca = PCA(k=2, inputCol="weather_features", outputCol="pca_features")

# Stage 7: Rename Power_Zone_3; label (for Spark ML estimators)
label_transform = SQLTransformer(
    statement="SELECT *, Power_Zone_3 AS label FROM __THIS__"
)

# Stage 8: Assemble final feature vector
final_assembler = VectorAssembler(
    inputCols=["pca_features", "Hour_binary",
               "Power_Zone_1", "Power_Zone_2", "Month_ohe"],
    outputCol="features"
)

####1.3: Build pipeline

In [20]:
# Chain all stages into a single Pipeline
pipeline = Pipeline(stages=[
    sql_transform,
    binarizer,
    month_indexer,
    month_encoder,
    weather_assembler,
    pca,
    label_transform,
    final_assembler
])

####1.4 Set up cross-validation

In [21]:
# Elastic net linear regression
lr = LinearRegression(featuresCol="features", labelCol="label")

# All 121 combos of these values will be tried
param_values = [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]

param_grid = ParamGridBuilder() \
    .addGrid(lr.regParam, param_values) \
    .addGrid(lr.elasticNetParam, param_values) \
    .build()

# RMSE as error metric
evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

# 5-fold cross-validation over full pipeline + LR model
cv = CrossValidator(
    estimator=Pipeline(stages=pipeline.getStages() + [lr]),
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=5,
    seed=42
)

####1.5: Fit the model

In [22]:
# Fit the model
print("Fitting model")
cv_model = cv.fit(spark_df)
print("Done")

Fitting model
Done


####1.6: Report results

In [23]:
# Extract best model and report metrics
best_model = cv_model.bestModel
best_lr = best_model.stages[-1]

print(f"Best regParam:         {best_lr.getRegParam()}")
print(f"Best elasticNetParam:  {best_lr.getElasticNetParam()}")

cv_rmse = min(cv_model.avgMetrics)
print(f"\nCV RMSE (best):        {cv_rmse:.4f}")

# Evaluate on full training set
train_predictions = best_model.transform(spark_df)
train_rmse = evaluator.evaluate(train_predictions)
print(f"Training RMSE:         {train_rmse:.4f}")

Best regParam:         0.5
Best elasticNetParam:  0.99

CV RMSE (best):        2147.6978
Training RMSE:         2147.0984


####1.7: Residuals table

In [24]:
# Compute residuals and display results
results_df = train_predictions.withColumn(
    "residual", col("label") - col("prediction")
)

results_df.select("label", "prediction", "residual").show(20)

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386| 20878.78254403584|-637.8186840358394|
|20131.08434|18656.527071209268|1474.5572687907334|
|19668.43373|18201.099586882665|1467.3341431173358|
|18899.27711|17587.174171164934|1312.1029388350653|
|18442.40964|16993.896273432933|1448.5133665670692|
|18130.12048|16514.383329831748|1615.7371501682537|
|17945.06024| 16090.04813906499|1855.0121009350096|
|17459.27711|15719.560714142199|1739.7163958578003|
|17025.54217|15268.009309101113|1757.5328608988875|
|16794.21687| 14935.36343057604|1858.8534394239596|
|16638.07229|14649.636731294482| 1988.435558705518|
|16395.18072|14412.201904580645|1982.9788154193557|
|16117.59036|14080.052711333032|2037.5376486669684|
| 15822.6506|13622.161876861775|2200.4887231382254|
|15672.28916|13447.751522172424| 2224.537637827576|
|15597.10843|13299.823707383453| 2297.284722616547|
|15510.36145

#### Section summary

The elastic net model was fit using 5-fold cross-validation across a grid of 121 parameter combinations (11 values each for regParam and elasticNetParam). The best parameters chosen by CV minimized the average RMSE across folds. The training RMSE gives us a sense of how well the model fits the data it was trained on, while the CV RMSE is a better reflection of how we'd expect the model to perform on unseen data. We'd expect it to be somewhat higher than the training RMSE since CV intentionally withholds portions of the data during fitting.
The transformation pipeline played an important role in preparing the data for modeling. Binarizing the Hour column captures a meaningful distinction between nighttime and daytime power consumption patterns without treating hour as a continuous variable. One-hot encoding Month allows the model to account for seasonal variation across the year. The PCA step reduces the five correlated weather-related variables (Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and Diffuse_Flows) down to two principal components, which helps reduce multicollinearity and keeps the feature space manageable.
The residuals table above shows the difference between the actual Power_Zone_3 values and the model's predictions. Residuals close to zero indicate a good fit for that observation. Larger residuals, whether positive or negative, point to cases where the model struggled, which could be due to unusual weather conditions, atypical consumption patterns, or information not captured by the available predictors.

### Part 2: Streaming

**Note:** Since google colab can't run 2 things in the same notebook, we must use 2 notebooks/browser tabs simultaneously. This notebook, EMartinez_Project3.ipynb, will be referred to as the primary notebook, while the other notebook, Project3_Producer.ipynb, will be referred to as the producer notebook. The primary notebook runs the stream reader, while the producer notebook runs the producer. They share the same google drive folder.

####2.1: Define stream schema

In [26]:
# Schema to match csv columns
stream_schema = StructType([
    StructField("Temperature",           DoubleType(), True),
    StructField("Humidity",              DoubleType(), True),
    StructField("Wind_Speed",            DoubleType(), True),
    StructField("General_Diffuse_Flows", DoubleType(), True),
    StructField("Diffuse_Flows",         DoubleType(), True),
    StructField("Power_Zone_1",          DoubleType(), True),
    StructField("Power_Zone_2",          DoubleType(), True),
    StructField("Power_Zone_3",          DoubleType(), True),
    StructField("Month",                 StringType(), True),
    StructField("Hour",                  IntegerType(), True),
])

####2.2: Setup readstream

In [27]:
# Watch stream_input/ for csv files
stream_df = spark.readStream \
    .schema(stream_schema) \
    .option("header", True) \
    .csv(f"{BASE}/stream_input/")

####2.3: Apply model to stream

In [28]:
# Run predictions on incoming stream
stream_predictions = best_model.transform(stream_df) \
    .withColumn("residual", col("label") - col("prediction")) \
    .select("label", "prediction", "residual")

# Second transformation: rename Power_Zone_3, label
stream_labeled = stream_df.withColumnRenamed("Power_Zone_3", "label")

# Join both on label
joined_stream = stream_predictions.join(stream_labeled, on="label")